Câu 3: Cho cây có trọng số trên đỉnh và trọng số trên cạnh. Cần dùng thuật toán UCS để tìm đường đi từ `S` đến một đỉnh có trọng số trên đỉnh bằng `0`.


a)	Hãy viết đoạn code biểu diễn cây bằng tập đỉnh `V`, trọng số heuristic trên đỉnh `H` và tập cạnh có chi phí `E`.

Dán code:

In [1]:
V = ["S", "A", "B", "C", "D", "E", "F", "G", "H", "K", "M", "N", "I", "J", "L", "Z"]

H = {
    "S": 10,
    "A": 9,
    "B": 8,
    "C": 7,
    "D": 6,
    "E": 5,
    "F": 4,
    "G": 10,
    "H": 10,
    "K": 3,
    "M": 0,
    "N": 10,
    "I": 6,
    "J": 0,
    "L": 9,
    "Z": 8,
}

E = [
    ("S", "A", 5),
    ("S", "B", 6),
    ("S", "C", 5),
    ("A", "D", 6),
    ("A", "E", 7),
    ("D", "M", 5),
    ("D", "N", 8),
    ("E", "I", 8),
    ("B", "F", 3),
    ("B", "G", 4),
    ("F", "J", 4),
    ("F", "L", 4),
    ("C", "H", 6),
    ("C", "K", 4),
    ("K", "Z", 2),
]

class Graph:
    def __init__(self):
        self.adjacency = {}
        self.heuristic = {}

    def add_node(self, node, heuristic_value):
        if node not in self.adjacency:
            self.adjacency[node] = []
        self.heuristic[node] = heuristic_value

    def add_edge(self, u, v, cost):
        self.adjacency[u].append((v, cost))
        self.adjacency[v].append((u, cost))

    def sort_neighbors(self, vertex_order):
        for node in self.adjacency:
            self.adjacency[node].sort(key=lambda item: vertex_order[item[0]])


def build_graph(vertices, edges, heuristic_values):
    graph = Graph()

    for vertex in vertices:
        graph.add_node(vertex, heuristic_values[vertex])

    for u, v, cost in edges:
        graph.add_edge(u, v, cost)

    vertex_order = {vertex: index for index, vertex in enumerate(vertices)}
    graph.sort_neighbors(vertex_order)
    return graph


def format_adjacency(graph):
    result = []

    for vertex in V:
        neighbors = ", ".join(
            f"{neighbor}(cost={cost})"
            for neighbor, cost in graph.adjacency[vertex]
        )
        result.append(f"{vertex}: {neighbors}")

    return "\n".join(result)


def main():
    graph = build_graph(V, E, H)

    print("Tap dinh V:")
    print(V)
    print()
    print("Trong so tren dinh H:")
    for vertex in V:
        print(f"h({vertex}) = {H[vertex]}")
    print()
    print("Tap canh E:")
    print(E)
    print()
    print("Danh sach ke:")
    print(format_adjacency(graph))


if __name__ == "__main__":
    main()

Tap dinh V:
['S', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'K', 'M', 'N', 'I', 'J', 'L', 'Z']

Trong so tren dinh H:
h(S) = 10
h(A) = 9
h(B) = 8
h(C) = 7
h(D) = 6
h(E) = 5
h(F) = 4
h(G) = 10
h(H) = 10
h(K) = 3
h(M) = 0
h(N) = 10
h(I) = 6
h(J) = 0
h(L) = 9
h(Z) = 8

Tap canh E:
[('S', 'A', 5), ('S', 'B', 6), ('S', 'C', 5), ('A', 'D', 6), ('A', 'E', 7), ('D', 'M', 5), ('D', 'N', 8), ('E', 'I', 8), ('B', 'F', 3), ('B', 'G', 4), ('F', 'J', 4), ('F', 'L', 4), ('C', 'H', 6), ('C', 'K', 4), ('K', 'Z', 2)]

Danh sach ke:
S: A(cost=5), B(cost=6), C(cost=5)
A: S(cost=5), D(cost=6), E(cost=7)
B: S(cost=6), F(cost=3), G(cost=4)
C: S(cost=5), H(cost=6), K(cost=4)
D: A(cost=6), M(cost=5), N(cost=8)
E: A(cost=7), I(cost=8)
F: B(cost=3), J(cost=4), L(cost=4)
G: B(cost=4)
H: C(cost=6)
K: C(cost=4), Z(cost=2)
M: D(cost=5)
N: D(cost=8)
I: E(cost=8)
J: F(cost=4)
L: F(cost=4)
Z: K(cost=2)


### Giải thích câu a

Ở câu a, em biểu diễn cây bằng các thành phần:

- `V` là tập đỉnh của cây.
- `H` là trọng số heuristic trên từng đỉnh. Đỉnh đích là các đỉnh có `h = 0`, trong bài này gồm `M` và `J`.
- `E` là tập cạnh có trọng số, mỗi cạnh được lưu dưới dạng `(u, v, cost)`.

UCS cần trọng số cạnh để tính chi phí thật từ `S` đến từng đỉnh. Vì vậy mỗi cạnh trong `E` đều có chi phí đi kèm.

Vì cây được xem là vô hướng nên khi thêm cạnh, chương trình thêm cả hai chiều vào danh sách kề. Class `Graph` lưu `adjacency` để phục vụ duyệt cây và `heuristic` để kiểm tra đỉnh nào là đích.


b)	Hãy viết chương trình sử dụng thuật toán UCS để tìm đường đi từ `S` đến một đỉnh có `h = 0`. Trong chương trình, hãy in ra thứ tự đỉnh khám phá và bảng trạng thái frontier.

Dán code:

In [2]:
import heapq


V = ["S", "A", "B", "C", "D", "E", "F", "G", "H", "K", "M", "N", "I", "J", "L", "Z"]

H = {
    "S": 10, "A": 9, "B": 8, "C": 7, "D": 6, "E": 5, "F": 4, "G": 10,
    "H": 10, "K": 3, "M": 0, "N": 10, "I": 6, "J": 0, "L": 9, "Z": 8,
}

E = [
    ("S", "A", 5), ("S", "B", 6), ("S", "C", 5), ("A", "D", 6),
    ("A", "E", 7), ("D", "M", 5), ("D", "N", 8), ("E", "I", 8),
    ("B", "F", 3), ("B", "G", 4), ("F", "J", 4), ("F", "L", 4),
    ("C", "H", 6), ("C", "K", 4), ("K", "Z", 2),
]


class Graph:
    def __init__(self):
        self.adjacency = {}
        self.heuristic = {}

    def add_node(self, node, heuristic_value):
        if node not in self.adjacency:
            self.adjacency[node] = []
        self.heuristic[node] = heuristic_value

    def add_edge(self, u, v, cost):
        self.adjacency[u].append((v, cost))
        self.adjacency[v].append((u, cost))

    def sort_neighbors(self, vertex_order):
        for node in self.adjacency:
            self.adjacency[node].sort(key=lambda item: vertex_order[item[0]])


def build_graph(vertices, edges, heuristic_values):
    graph = Graph()
    for vertex in vertices:
        graph.add_node(vertex, heuristic_values[vertex])
    for u, v, cost in edges:
        graph.add_edge(u, v, cost)
    vertex_order = {vertex: index for index, vertex in enumerate(vertices)}
    graph.sort_neighbors(vertex_order)
    return graph


def reconstruct_path(parent, start, goal):
    path = []
    current = goal
    while current is not None:
        path.append(current)
        current = parent[current]
    path.reverse()
    return path if path[0] == start else None


def ucs_to_zero_heuristic_goal(graph, start):
    frontier = []
    order = 0
    heapq.heappush(frontier, (0, order, start))
    parent = {start: None}
    cost_so_far = {start: 0}
    visited = set()
    exploration_order = []
    steps = []

    while frontier:
        frontier_before = list(frontier)
        current_cost, _, current = heapq.heappop(frontier)
        if current in visited:
            continue
        visited.add(current)
        exploration_order.append(current)

        if graph.heuristic[current] == 0:
            path = reconstruct_path(parent, start, current)
            steps.append({
                "frontier_before": frontier_before,
                "current": current,
                "g": current_cost,
                "h": graph.heuristic[current],
                "added": [],
                "frontier_after": list(frontier),
                "note": "Dung vi h = 0",
            })
            return path, current, current_cost, exploration_order, steps

        added = []
        for neighbor, edge_cost in graph.adjacency[current]:
            if neighbor in visited:
                continue
            new_cost = current_cost + edge_cost
            if neighbor not in cost_so_far or new_cost < cost_so_far[neighbor]:
                cost_so_far[neighbor] = new_cost
                parent[neighbor] = current
                order += 1
                heapq.heappush(frontier, (new_cost, order, neighbor))
                added.append((neighbor, new_cost))

        steps.append({
            "frontier_before": frontier_before,
            "current": current,
            "g": current_cost,
            "h": graph.heuristic[current],
            "added": added,
            "frontier_after": list(frontier),
            "note": "",
        })

    return None, None, None, exploration_order, steps


def format_frontier_ucs(frontier):
    ordered = sorted(frontier)
    return "[" + ", ".join(f"{node}(g={cost})" for cost, _, node in ordered) + "]"


def make_table(headers, rows):
    widths = [
        max(len(str(row[index])) for row in [headers] + rows)
        for index in range(len(headers))
    ]
    separator = "+" + "+".join("-" * (width + 2) for width in widths) + "+"

    def format_row(row):
        cells = [
            str(value).ljust(widths[index])
            for index, value in enumerate(row)
        ]
        return "| " + " | ".join(cells) + " |"

    lines = [separator, format_row(headers), separator]
    lines.extend(format_row(row) for row in rows)
    lines.append(separator)
    return "\n".join(lines)


def format_steps(steps):
    headers = ["Buoc", "Frontier truoc", "Lay ra", "Them vao", "Frontier sau"]
    rows = []
    for index, step in enumerate(steps, start=1):
        added = ", ".join(f"{node}(g={cost})" for node, cost in step["added"]) if step["added"] else "Khong co"
        current = f"{step['current']}(g={step['g']}, h={step['h']})"
        rows.append([
            index,
            format_frontier_ucs(step["frontier_before"]),
            current,
            added,
            format_frontier_ucs(step["frontier_after"]),
        ])
    return "\n".join([
        "Bang trang thai:",
        make_table(headers, rows),
        "Dung khi lay ra dinh co h = 0.",
    ])


def solve():
    graph = build_graph(V, E, H)
    path, goal, total_cost, exploration_order, steps = ucs_to_zero_heuristic_goal(graph, "S")
    lines = ["THUAT TOAN UCS", "Thu tu dinh kham pha:", " -> ".join(exploration_order), "", format_steps(steps), ""]
    if path is None:
        lines.append("Khong tim thay duong di")
    else:
        lines.extend([
            f"Dinh dich tim duoc: {goal}",
            f"Tong chi phi duong di: {total_cost}",
            "Duong di tu S den dinh co h = 0:",
            " -> ".join(path),
        ])
    return "\n".join(lines)


def main():
    print(solve())


if __name__ == "__main__":
    main()


THUAT TOAN UCS
Thu tu dinh kham pha:
S -> A -> C -> B -> K -> F -> G -> D -> H -> Z -> E -> J

Bang trang thai:
+------+-----------------------------------------------------------------+---------------+------------------------+-----------------------------------------------------------------+
| Buoc | Frontier truoc                                                  | Lay ra        | Them vao               | Frontier sau                                                    |
+------+-----------------------------------------------------------------+---------------+------------------------+-----------------------------------------------------------------+
| 1    | [S(g=0)]                                                        | S(g=0, h=10)  | A(g=5), B(g=6), C(g=5) | [A(g=5), C(g=5), B(g=6)]                                        |
| 2    | [A(g=5), C(g=5), B(g=6)]                                        | A(g=5, h=9)   | D(g=11), E(g=12)       | [C(g=5), B(g=6), D(g=11), E(g=12)]          

### Giải thích tác dụng của từng hàm

- `Graph.__init__()`: khởi tạo cây rỗng, gồm `adjacency` để lưu danh sách kề và `heuristic` để lưu giá trị `h(n)` của từng đỉnh.
- `Graph.add_node(node, heuristic_value)`: thêm một đỉnh vào cây và lưu giá trị heuristic của đỉnh đó. Nếu đỉnh chưa có trong danh sách kề thì tạo danh sách kề rỗng cho đỉnh.
- `Graph.add_edge(u, v, cost)`: thêm cạnh có chi phí giữa hai đỉnh `u` và `v`. Vì cây được biểu diễn dạng vô hướng nên cạnh được lưu ở cả hai chiều.
- `Graph.sort_neighbors(vertex_order)`: sắp xếp các đỉnh kề theo thứ tự xuất hiện trong tập đỉnh `V`, giúp thứ tự duyệt ổn định và dễ kiểm tra.
- `build_graph(vertices, edges, heuristic_values)`: tạo đối tượng `Graph`, thêm toàn bộ đỉnh, thêm toàn bộ cạnh, sắp xếp danh sách kề rồi trả về cây hoàn chỉnh.
- `reconstruct_path(parent, start, goal)`: truy vết đường đi từ đỉnh đích về đỉnh bắt đầu dựa trên dictionary `parent`, sau đó đảo ngược để lấy đường đi đúng chiều từ `start` đến `goal`.
- `ucs_to_zero_heuristic_goal(graph, start)`: thực hiện UCS từ `start` để tìm đỉnh đầu tiên có `h = 0`. Hàm dùng priority queue, luôn mở rộng đỉnh có tổng chi phí nhỏ nhất, cập nhật đường đi rẻ hơn và dừng khi lấy được đỉnh đích khỏi frontier.
- `format_frontier_ucs(frontier)`: định dạng hàng đợi ưu tiên của UCS, hiển thị các đỉnh đang chờ xét cùng tổng chi phí từ `S` đến đỉnh đó.
- `format_list(values)`: chuyển một danh sách giá trị thành chuỗi để in ra màn hình theo định dạng dễ đọc.
- `make_table(headers, rows)`: tạo bảng văn bản từ tiêu đề cột và các dòng dữ liệu.
- `format_row(row)`: hàm phụ dùng khi tạo bảng, giúp căn chỉnh dữ liệu của từng dòng.
- `format_steps(steps)`: chuyển danh sách các bước chạy thuật toán thành bảng trạng thái để người đọc theo dõi quá trình duyệt.
- `solve()`: xây dựng cây, chạy UCS, gom bảng bước chạy, thứ tự khám phá, đường đi và tổng chi phí thành chuỗi kết quả hoàn chỉnh.
- `main()`: gọi `solve()` và in kết quả ra màn hình.

### Giải thích câu b

Ở câu b, em sử dụng thuật toán UCS để tìm đường đi từ `S` đến một đỉnh có heuristic bằng `0`.

UCS luôn chọn đỉnh có chi phí thật `g(n)` nhỏ nhất để khám phá trước. Thuật toán không dùng heuristic để chọn đường đi, nhưng vẫn dùng `H` để kiểm tra đỉnh nào là đích.

### Công thức / nguyên tắc sử dụng

Chi phí thật từ `S` đến một đỉnh `n` được ký hiệu là:

```text
g(n) = tổng chi phí cạnh từ S đến n
```

Khi đi từ đỉnh `u` sang đỉnh kề `v`, chi phí mới được tính:

```text
g(v) = g(u) + cost(u, v)
```

Mỗi phần tử trong hàng đợi ưu tiên có dạng:

```text
(current_cost, order, node)
```

Trong đó:

- `current_cost` là chi phí thật từ `S` đến `node`.
- `order` giúp ổn định thứ tự khi nhiều đỉnh có cùng chi phí.
- `node` là đỉnh đang chờ khám phá.

Điều kiện dừng của bài này là:

```text
h(current) = 0
```

Khi lấy ra một đỉnh có `h = 0`, đường đi đến đỉnh đó là đường có tổng chi phí nhỏ nhất trong các đỉnh đích có thể gặp theo UCS.

Bảng trạng thái cho biết frontier trước mỗi bước, đỉnh được lấy ra, các đỉnh được thêm vào frontier và frontier sau bước đó.


### Nhận xét về kết quả câu b

Thứ tự đỉnh khám phá của UCS là:

```text
S -> A -> C -> B -> K -> F -> G -> D -> H -> Z -> E -> J
```

Thuật toán dừng tại `J` vì `h(J) = 0`.

Đường đi tìm được là:

```text
S -> B -> F -> J
```

Tổng chi phí của đường đi này là:

```text
6 + 3 + 4 = 13
```

UCS tìm `J` trước `M` vì chi phí đến `J` là `13`, nhỏ hơn chi phí đến `M` là `16`. Vì UCS ưu tiên chi phí thật, kết quả là đường đi có tổng chi phí nhỏ nhất trong lần tìm kiếm này.


Dán kết quả thực thi:

```text
THUAT TOAN UCS
Thu tu dinh kham pha:
S -> A -> C -> B -> K -> F -> G -> D -> H -> Z -> E -> J

Bang trang thai:
+------+-----------------------------------------------------------------+---------------+------------------------+-----------------------------------------------------------------+
| Buoc | Frontier truoc                                                  | Lay ra        | Them vao               | Frontier sau                                                    |
+------+-----------------------------------------------------------------+---------------+------------------------+-----------------------------------------------------------------+
| 1    | [S(g=0)]                                                        | S(g=0, h=10)  | A(g=5), B(g=6), C(g=5) | [A(g=5), C(g=5), B(g=6)]                                        |
| 2    | [A(g=5), C(g=5), B(g=6)]                                        | A(g=5, h=9)   | D(g=11), E(g=12)       | [C(g=5), B(g=6), D(g=11), E(g=12)]                              |
| 3    | [C(g=5), B(g=6), D(g=11), E(g=12)]                              | C(g=5, h=7)   | H(g=11), K(g=9)        | [B(g=6), K(g=9), D(g=11), H(g=11), E(g=12)]                     |
| 4    | [B(g=6), K(g=9), D(g=11), H(g=11), E(g=12)]                     | B(g=6, h=8)   | F(g=9), G(g=10)        | [K(g=9), F(g=9), G(g=10), D(g=11), H(g=11), E(g=12)]            |
| 5    | [K(g=9), F(g=9), G(g=10), D(g=11), H(g=11), E(g=12)]            | K(g=9, h=3)   | Z(g=11)                | [F(g=9), G(g=10), D(g=11), H(g=11), Z(g=11), E(g=12)]           |
| 6    | [F(g=9), G(g=10), D(g=11), H(g=11), Z(g=11), E(g=12)]           | F(g=9, h=4)   | J(g=13), L(g=13)       | [G(g=10), D(g=11), H(g=11), Z(g=11), E(g=12), J(g=13), L(g=13)] |
| 7    | [G(g=10), D(g=11), H(g=11), Z(g=11), E(g=12), J(g=13), L(g=13)] | G(g=10, h=10) | Khong co               | [D(g=11), H(g=11), Z(g=11), E(g=12), J(g=13), L(g=13)]          |
| 8    | [D(g=11), H(g=11), Z(g=11), E(g=12), J(g=13), L(g=13)]          | D(g=11, h=6)  | M(g=16), N(g=19)       | [H(g=11), Z(g=11), E(g=12), J(g=13), L(g=13), M(g=16), N(g=19)] |
| 9    | [H(g=11), Z(g=11), E(g=12), J(g=13), L(g=13), M(g=16), N(g=19)] | H(g=11, h=10) | Khong co               | [Z(g=11), E(g=12), J(g=13), L(g=13), M(g=16), N(g=19)]          |
| 10   | [Z(g=11), E(g=12), J(g=13), L(g=13), M(g=16), N(g=19)]          | Z(g=11, h=8)  | Khong co               | [E(g=12), J(g=13), L(g=13), M(g=16), N(g=19)]                   |
| 11   | [E(g=12), J(g=13), L(g=13), M(g=16), N(g=19)]                   | E(g=12, h=5)  | I(g=20)                | [J(g=13), L(g=13), M(g=16), N(g=19), I(g=20)]                   |
| 12   | [J(g=13), L(g=13), M(g=16), N(g=19), I(g=20)]                   | J(g=13, h=0)  | Khong co               | [L(g=13), M(g=16), N(g=19), I(g=20)]                            |
+------+-----------------------------------------------------------------+---------------+------------------------+-----------------------------------------------------------------+
Dung khi lay ra dinh co h = 0.

Dinh dich tim duoc: J
Tong chi phi duong di: 13
Duong di tu S den dinh co h = 0:
S -> B -> F -> J
```